In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("RDD Lab").master("local[*]").getOrCreate()
sc = spark.sparkContext

In [2]:
raw = sc.textFile("/user/itvlab/rdd_lab/employees.txt")
raw.collect()

['emp_id,name,department,job_title,salary,location,hire_date,performance_rating,years_exp',
 '1,John Smith,Engineering,Senior Developer,125000,San Francisco,2021-03-15,4.5,8',
 '2,Sarah Johnson,Sales,Account Executive,85000,New York,2022-01-10,4.2,5',
 '3,Michael Williams,Engineering,Software Engineer,95000,Austin,2023-06-20,3.8,3',
 '4,Jennifer Brown,Marketing,Marketing Manager,92000,Chicago,2020-11-05,4.7,7',
 '',
 '5,David Jones,Finance,Senior Analyst,105000,Boston,2021-08-12,4.3,6',
 '6,Lisa Garcia,IT,DevOps Engineer,115000,Seattle,2022-04-18,4.6,4',
 '7,Robert Martinez,Legal,Legal Counsel145000,San Francisco,2019-09-22,4.8,10, 5',
 '8,Patricia Wilson,HR,HR Manager,88000,Denver,2020-02-28,4.1,6',
 '',
 '9,James Anderson,Sales,Sales Manager,110000,Atlanta,2021-12-01,4.4,7',
 '10,Mary Thomas,Engineering,Tech Lead,145000,Seattle,2018-07-14,4.9,12']

In [3]:
header = raw.first()
data = raw.filter(lambda line: line != header and line.strip() != "")

parsed = data.map(lambda line: line.split(","))

for row in parsed.collect():
    print(row)

['1', 'John Smith', 'Engineering', 'Senior Developer', '125000', 'San Francisco', '2021-03-15', '4.5', '8']
['2', 'Sarah Johnson', 'Sales', 'Account Executive', '85000', 'New York', '2022-01-10', '4.2', '5']
['3', 'Michael Williams', 'Engineering', 'Software Engineer', '95000', 'Austin', '2023-06-20', '3.8', '3']
['4', 'Jennifer Brown', 'Marketing', 'Marketing Manager', '92000', 'Chicago', '2020-11-05', '4.7', '7']
['5', 'David Jones', 'Finance', 'Senior Analyst', '105000', 'Boston', '2021-08-12', '4.3', '6']
['6', 'Lisa Garcia', 'IT', 'DevOps Engineer', '115000', 'Seattle', '2022-04-18', '4.6', '4']
['7', 'Robert Martinez', 'Legal', 'Legal Counsel145000', 'San Francisco', '2019-09-22', '4.8', '10', ' 5']
['8', 'Patricia Wilson', 'HR', 'HR Manager', '88000', 'Denver', '2020-02-28', '4.1', '6']
['9', 'James Anderson', 'Sales', 'Sales Manager', '110000', 'Atlanta', '2021-12-01', '4.4', '7']
['10', 'Mary Thomas', 'Engineering', 'Tech Lead', '145000', 'Seattle', '2018-07-14', '4.9', '12']


In [4]:
names = parsed.map(lambda r: (r[1], 1))
name_counts = names.reduceByKey(lambda a, b: a + b)

name_counts.collect()

[('Sarah Johnson', 1),
 ('Michael Williams', 1),
 ('Jennifer Brown', 1),
 ('David Jones', 1),
 ('Robert Martinez', 1),
 ('Patricia Wilson', 1),
 ('James Anderson', 1),
 ('Mary Thomas', 1),
 ('John Smith', 1),
 ('Lisa Garcia', 1)]

In [5]:
def is_valid(line):
    parts = line.split(",")
    if len(parts) != 9:
        return False
    try:
        int(parts[0])
        float(parts[4])
        float(parts[7])
        int(parts[8])
        return True
    except:
        return False

valid = data.filter(is_valid)
invalid = data.filter(lambda line: not is_valid(line))

print("Valid:")
for r in valid.collect():
    print(r)

print("Invalid:")
for r in invalid.collect():
    print(r)

Valid:
1,John Smith,Engineering,Senior Developer,125000,San Francisco,2021-03-15,4.5,8
2,Sarah Johnson,Sales,Account Executive,85000,New York,2022-01-10,4.2,5
3,Michael Williams,Engineering,Software Engineer,95000,Austin,2023-06-20,3.8,3
4,Jennifer Brown,Marketing,Marketing Manager,92000,Chicago,2020-11-05,4.7,7
5,David Jones,Finance,Senior Analyst,105000,Boston,2021-08-12,4.3,6
6,Lisa Garcia,IT,DevOps Engineer,115000,Seattle,2022-04-18,4.6,4
8,Patricia Wilson,HR,HR Manager,88000,Denver,2020-02-28,4.1,6
9,James Anderson,Sales,Sales Manager,110000,Atlanta,2021-12-01,4.4,7
10,Mary Thomas,Engineering,Tech Lead,145000,Seattle,2018-07-14,4.9,12
Invalid:
7,Robert Martinez,Legal,Legal Counsel145000,San Francisco,2019-09-22,4.8,10, 5


In [6]:
valid_parsed = valid.map(lambda line: line.split(","))

dept_salary = valid_parsed.map(lambda r: (r[2], (float(r[4]), 1)))
dept_totals = dept_salary.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
dept_avg = dept_totals.mapValues(lambda x: x[0] / x[1])

dept_avg.collect()

[('Sales', 97500.0),
 ('Finance', 105000.0),
 ('Engineering', 121666.66666666667),
 ('Marketing', 92000.0),
 ('IT', 115000.0),
 ('HR', 88000.0)]

In [7]:
dept_count = valid_parsed.map(lambda r: (r[2], 1)).reduceByKey(lambda a, b: a + b)

dept_count.collect()

[('Sales', 2),
 ('Finance', 1),
 ('Engineering', 3),
 ('Marketing', 1),
 ('IT', 1),
 ('HR', 1)]

In [8]:
spark.stop()